In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
import joblib
import os
import json

print(f'Rebuilding data pipeline...')
MODELS_DIR = '../outputs/models'
OUTPUT_DIR = '../outputs/figures'
 
BG    = '#ffffff'
MUTED = '#888888'
TEXT  = '#222222'
WELL_COLOURS = {
    '15/9-F-1 C':  '#f39c12',
    '15/9-F-11':   '#3498db',
    '15/9-F-12':   '#2ecc71',
    '15/9-F-14':   '#e74c3c',
    '15/9-F-15 D': '#9b59b6',
    '15/9-F-5':    '#1abc9c',
}
AGG_RULES = {
    'OIL_RATE_NORM':         'mean',
    'AVG_WHP_P':             'mean',
    'AVG_DOWNHOLE_PRESSURE': 'mean',
    'WCT':                   'mean',
    'GOR':                   'median',
    'DRAWDOWN':              'mean',
    'DAYS_ON_PROD':          'max',
    'ON_STREAM_HRS':         'sum',
}

df = pd.read_parquet('../data/processed/cleaned.parquet')
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'])
df = df.set_index('DATEPRD').sort_index()
df['OIL_RATE_NORM'] = df.groupby('NPD_WELL_BORE_NAME')['OIL_RATE_NORM'].transform(
    lambda x : x.clip(upper= x.quantile(0.99))
)

def style_ax(ax, title, xlabel='', ylabel='', fontsize=11):
    ax.set_title(title, fontsize=fontsize, fontweight='bold', color=TEXT, pad=10)
    ax.set_xlabel(xlabel, fontsize=10, color=MUTED)
    ax.set_ylabel(ylabel, fontsize=10, color=MUTED)
    ax.tick_params(colors=MUTED, labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor('#dddddd')

ARPS_BENCHMARK = {
    '15/9-F-12': {'RMSE': 307.8, 'MAE': 260.4, 'R2': -1.420},
    '15/9-F-14': {'RMSE': 203.9, 'MAE': 203.6, 'R2': -10.703},
}
 
XGB_RESULTS = {
    '15/9-F-12': {'RMSE': 572.5, 'MAE': 523.0, 'R2': -7.372},
    '15/9-F-14': {'RMSE': 170.4, 'MAE': 160.1, 'R2': -7.180},
}

TARGET_WELLS = ['15/9-F-12', '15/9-F-14']
monthly_well_data = {}
for well_name,well_df in df.groupby('NPD_WELL_BORE_NAME'):
    if(well_name not in TARGET_WELLS):
        print(f'this well has no clean data')
        continue
    cols_availble = {k: v for k, v in AGG_RULES.items() if k in well_df.columns}
    monthly = well_df[list(cols_availble.keys())].resample('ME').agg(
        cols_availble
    ).dropna(subset=['OIL_RATE_NORM'])
    monthly_well_data[well_name] = monthly

def engineer_features(month_df):
    df_feature = month_df.copy()
    df_feature['lag_1_oil'] = df_feature['OIL_RATE_NORM'].shift(1)
    df_feature['lag_3_oil'] = df_feature['OIL_RATE_NORM'].shift(3)
    df_feature['lag_6_oil'] = df_feature['OIL_RATE_NORM'].shift(6)
    df_feature['lag_1_wct'] = df_feature['WCT'].shift(1) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_3_wct'] = df_feature['WCT'].shift(3) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_6_wct'] = df_feature['WCT'].shift(6) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_1_whp'] = df_feature['AVG_WHP_P'].shift(1) if 'AVG_WHP_P' in df_feature.columns else np.nan
    df_feature['rolling_3m'] = df_feature['OIL_RATE_NORM'].rolling(window=3, min_periods=2).mean()
    df_feature['rolling_6m'] = df_feature['OIL_RATE_NORM'].rolling(window=6, min_periods=3).mean()
    df_feature['month_over_month'] = df_feature['OIL_RATE_NORM'].pct_change().clip(-1, 1)
    df_feature['rate_vs_6m_avg'] = df_feature['lag_1_oil'] / (df_feature['rolling_6m'] + 1e-6)
    df_feature['wct_change'] = df_feature['WCT'].diff()
    df_feature['well_age'] = df_feature['DAYS_ON_PROD'] / 365
    return df_feature

FEATURE_COLS = [
    'lag_1_oil', 'lag_3_oil', 'lag_6_oil',
    'lag_1_wct', 'lag_3_wct', 'lag_6_wct',
    'lag_1_whp', 'rolling_3m', 'rolling_6m',
    'month_over_month', 'rate_vs_6m_avg',
    'WCT', 'AVG_WHP_P', 'DAYS_ON_PROD',
    'AVG_DOWNHOLE_PRESSURE', 'DRAWDOWN', 'GOR',
    'wct_change', 'well_age',
]

TARGET_COL = 'OIL_RATE_NORM'

well_splits = {}
for well_name in TARGET_WELLS:
    if well_name not in monthly_well_data:
        continue
    monthly  = monthly_well_data[well_name]
    df_feat  = engineer_features(monthly).dropna(subset=[TARGET_COL])
    peak_idx = df_feat[TARGET_COL].idxmax()
    peak_pos = df_feat.index.get_loc(peak_idx)
    decline  = df_feat.iloc[peak_pos:].copy()
    n_train  = int(len(decline) * 0.8)
 
    feat_available = [c for c in FEATURE_COLS if c in decline.columns]
    train_clean = decline.iloc[:n_train][feat_available + [TARGET_COL]].dropna()
    test_clean  = decline.iloc[n_train:][feat_available + [TARGET_COL]].dropna()
 
    well_splits[well_name] = {
        'decline':         decline,
        'train_clean':     train_clean,
        'test_clean':      test_clean,
        'feat_available':  feat_available,
        'n_train':         n_train,
        'X_train': train_clean[feat_available],
        'y_train': train_clean[TARGET_COL],
        'X_test':  test_clean[feat_available],
        'y_test':  test_clean[TARGET_COL],
    }
 
print(f"Data rebuilt for {list(well_splits.keys())}")
 
 
print("\nLoading saved XGBoost models...")
models = {}
for well_name in TARGET_WELLS:
    model_path = f'{MODELS_DIR}/xgb_{well_name.replace("/", "-").replace(" ", "_")}.joblib'
    if os.path.exists(model_path):
        models[well_name] = joblib.load(model_path)
        print(f"  Loaded: {model_path}")
    else:
        print(f"  NOT FOUND: {model_path} — retrain from 03_xgboost.py first")

print("\nComputing SHAP values...")
shap_data = {}
 
for well_name in TARGET_WELLS:
    if well_name not in models or well_name not in well_splits:
        print(f"  Skipping {well_name} — model or data missing")
        continue
 
    model  = models[well_name]
    splits = well_splits[well_name]
 
    print(f"  Computing SHAP for {well_name}...")
 
    explainer = shap.TreeExplainer(model)
 
    shap_train = explainer(splits['X_train'])
 
    shap_test  = explainer(splits['X_test'])
 
    shap_data[well_name] = {
        'explainer':   explainer,
        'shap_train':  shap_train,   
        'shap_test':   shap_test,
        'X_train':     splits['X_train'],
        'X_test':      splits['X_test'],
        'y_train':     splits['y_train'],
        'y_test':      splits['y_test'],
    }
 
    print(f"    SHAP matrix shape (train): {shap_train.values.shape}")
    print(f"    Baseline (expected value): {explainer.expected_value:.1f} Sm³/day")
    print(f"    This is the average prediction across all training rows.")
 
print("\nPlot 09: Beeswarm plots...")
 
fig, axes = plt.subplots(1, len(shap_data), figsize=(16, 8), facecolor=BG)
if len(shap_data) == 1:
    axes = [axes]
 
for ax, (well_name, sd) in zip(axes, shap_data.items()):
    plt.sca(ax)
 
    shap.plots.beeswarm(
        sd['shap_train'],
        max_display=12,
        show=False,
        color_bar=True,
    )
 
    ax.set_title(f'SHAP Beeswarm — {well_name}\nGlobal feature impact on all training predictions',
                 fontsize=10, fontweight='bold', color=TEXT, pad=8)
    ax.tick_params(colors=MUTED, labelsize=8)
 
plt.tight_layout()
path = f'{OUTPUT_DIR}/09_shap_beeswarm.png'
fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print(f"  Saved → {path}")
 


 
print("\nPlot 10: Waterfall plots (local explanation of best and worst predictions)...")
 
for well_name, sd in shap_data.items():
    colour  = WELL_COLOURS.get(well_name, '#333333')
    y_test  = sd['y_test']
    X_test  = sd['X_test']
    model   = models[well_name]
 
    y_pred  = model.predict(X_test)
    errors  = np.abs(y_pred - y_test.values)
 
    worst_idx = np.argmax(errors)
    best_idx  = np.argmin(errors)
 
    fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)
    fig.suptitle(f'SHAP Waterfall — {well_name}\nHow XGBoost built each prediction from baseline',
                 fontsize=13, fontweight='bold', color=TEXT, y=1.01)
 
    for ax, idx, label in [
        (axes[0], worst_idx, f'WORST prediction\n(error={errors[worst_idx]:.0f} Sm³/day)'),
        (axes[1], best_idx,  f'BEST prediction\n(error={errors[best_idx]:.0f} Sm³/day)'),
    ]:
        plt.sca(ax)
 
        shap.plots.waterfall(
            sd['shap_test'][idx],
            max_display=10,
            show=False,
        )
 
        # Add actual vs predicted annotation
        actual_val = y_test.values[idx]
        pred_val   = y_pred[idx]
        date_str   = y_test.index[idx].strftime('%b %Y')
 
        ax.set_title(f'{label}\nDate: {date_str} | Actual: {actual_val:.0f} | Predicted: {pred_val:.0f}',
                     fontsize=9, fontweight='bold', color=TEXT, pad=6)
        ax.tick_params(colors=MUTED, labelsize=8)
 
    plt.tight_layout()
    safe_name = well_name.replace('/', '-').replace(' ', '_')
    path = f'{OUTPUT_DIR}/10_shap_waterfall_{safe_name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close()
    print(f"  Saved → {path}")

 
print("\nPlot 11: SHAP dependence plots...")
 
DEPENDENCE_FEATURES = ['lag_1_oil', 'WCT']
 
for well_name, sd in shap_data.items():
    n_features = len(DEPENDENCE_FEATURES)
    fig, axes = plt.subplots(1, n_features, figsize=(14, 5), facecolor=BG)
    if n_features == 1:
        axes = [axes]
 
    fig.suptitle(f'SHAP Dependence — {well_name}\nHow feature values affect predictions',
                 fontsize=12, fontweight='bold', color=TEXT)
 
    for ax, feat in zip(axes, DEPENDENCE_FEATURES):
        if feat not in sd['X_train'].columns:
            continue
 
        plt.sca(ax)
 
        shap.plots.scatter(
            sd['shap_train'][:, feat],
            color=sd['shap_train'],
            show=False,
            ax=ax,
        )
 
        ax.set_xlabel(feat, fontsize=10, color=MUTED)
        ax.set_ylabel(f'SHAP value for {feat}', fontsize=10, color=MUTED)
        ax.set_title(f'Effect of {feat} on predictions',
                     fontsize=10, fontweight='bold', color=TEXT)
        ax.tick_params(colors=MUTED, labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor('#dddddd')
 
    plt.tight_layout()
    safe_name = well_name.replace('/', '-').replace(' ', '_')
    path = f'{OUTPUT_DIR}/11_shap_dependence_{safe_name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close()
    print(f"  Saved → {path}")

print("\nPlot 12: Final summary dashboard...")
 
arps_json_path = f'{MODELS_DIR}/arps_params.json'
arps_params = {}
if os.path.exists(arps_json_path):
    with open(arps_json_path) as f:
        arps_params = json.load(f)
 
for well_name, sd in shap_data.items():
    colour  = WELL_COLOURS.get(well_name, '#333333')
    splits  = well_splits[well_name]
    model   = models[well_name]
 
    y_test  = splits['y_test']
    X_test  = splits['X_test']
    y_pred  = model.predict(X_test)
    n_train = splits['n_train']
 
    fig = plt.figure(figsize=(18, 8), facecolor=BG)
    fig.suptitle(f'Final Summary — {well_name}',
                 fontsize=14, fontweight='bold', color=TEXT, y=1.01)
 
    gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)
    ax_forecast = fig.add_subplot(gs[0, 0])
    ax_shap     = fig.add_subplot(gs[0, 1])
 
    monthly_all = monthly_well_data[well_name]['OIL_RATE_NORM']
    decline     = splits['decline']
 
    ax_forecast.plot(monthly_all.index, monthly_all.values,
                     color='#dddddd', linewidth=1, alpha=0.5,
                     label='All-time monthly actual')
 
    # Training period actual
    train_actual = decline.iloc[:n_train]['OIL_RATE_NORM']
    ax_forecast.plot(train_actual.index, train_actual.values,
                     color=colour, linewidth=2, alpha=0.8,
                     label='Actual (train)')
 
    # Test period actual
    ax_forecast.plot(y_test.index, y_test.values,
                     color=colour, linewidth=2.5, linestyle='--',
                     alpha=0.9, label='Actual (test)')
 
    # XGBoost predictions
    xgb_series = pd.Series(y_pred, index=y_test.index)
    xgb_rmse   = XGB_RESULTS.get(well_name, {}).get('RMSE', 0)
    ax_forecast.plot(xgb_series.index, xgb_series.values,
                     color='#3498db', linewidth=2.5, alpha=0.9,
                     label=f'XGBoost (RMSE={xgb_rmse:.0f})')
 
    # Arps forecast
    if well_name in arps_params:
        ap     = arps_params[well_name]
        winner = ap['winner']
        t_test = np.arange(n_train, n_train + len(y_test), dtype=float)
        if winner == 'exponential' and 'exponential_params' in ap:
            qi = ap['exponential_params']['qi']
            D  = ap['exponential_params']['D']
            arps_fc = qi * np.exp(-D * t_test)
        elif winner == 'hyperbolic' and 'hyperbolic_params' in ap:
            qi = ap['hyperbolic_params']['qi']
            D  = ap['hyperbolic_params']['D']
            b  = np.clip(ap['hyperbolic_params']['b'], 1e-5, 0.9999)
            arps_fc = qi / np.power(1.0 + b * D * t_test, 1.0 / b)
        else:
            arps_fc = None
 
        if arps_fc is not None:
            arps_rmse = ARPS_BENCHMARK.get(well_name, {}).get('RMSE', 0)
            ax_forecast.plot(y_test.index, arps_fc,
                             color='#f39c12', linewidth=2.5,
                             linestyle=':', alpha=0.9,
                             label=f'Arps {winner} (RMSE={arps_rmse:.0f})')
 
    # Train/test split line
    ax_forecast.axvline(y_test.index[0], color='#e74c3c',
                        linestyle='--', linewidth=1.2, alpha=0.5,
                        label='Train/test split')
 
    style_ax(ax_forecast,
             title='Production Forecast: Actual vs Arps vs XGBoost',
             xlabel='Date', ylabel='Oil Rate (Sm³/day)')
    ax_forecast.legend(fontsize=8, framealpha=0.4, loc='upper right')
 
  
    mean_abs_shap = pd.Series(
        np.abs(sd['shap_train'].values).mean(axis=0),
        index=splits['X_train'].columns
    ).sort_values(ascending=True).tail(12)  # top 12
 
    # Normalise to percentage
    mean_abs_shap_pct = mean_abs_shap / mean_abs_shap.sum() * 100
 
    bars = ax_shap.barh(
        mean_abs_shap_pct.index,
        mean_abs_shap_pct.values,
        color=colour, alpha=0.8,
        edgecolor='white', linewidth=0.5
    )
 
    for bar, val in zip(bars, mean_abs_shap_pct.values):
        if val > 1:
            ax_shap.text(val + 0.3,
                         bar.get_y() + bar.get_height() / 2,
                         f'{val:.1f}%', va='center',
                         fontsize=8, color=TEXT)
 
    style_ax(ax_shap,
             title='Mean |SHAP| — Feature importance\n(same units as target: Sm³/day)',
             xlabel='Mean |SHAP| importance (%)', ylabel='')
 
    plt.tight_layout()
    safe_name = well_name.replace('/', '-').replace(' ', '_')
    path = f'{OUTPUT_DIR}/12_final_dashboard_{safe_name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close()
    print(f"  Saved → {path}")


print("SHAP ANALYSIS COMPLETE")
 
for well_name, sd in shap_data.items():
    print(f"\n  {well_name}")
    print(f"  Baseline prediction: {sd['shap_train'].base_values[0]:.1f} Sm³/day")
    print(f"  (This is the average prediction across all training rows)")
 
    # Top 5 features by mean absolute SHAP
    mean_abs = pd.Series(
        np.abs(sd['shap_train'].values).mean(axis=0),
        index=well_splits[well_name]['X_train'].columns
    ).sort_values(ascending=False).head(5)
 
    print(f"  Top 5 features by mean |SHAP|:")
    for feat, val in mean_abs.items():
        print(f"    {feat:<25} avg impact: {val:.1f} Sm³/day")
 
print(f"\n  Plots saved:")
print(f"  09_shap_beeswarm.png          — global feature impact")
print(f"  10_shap_waterfall_[well].png  — best/worst prediction explained")
print(f"  11_shap_dependence_[well].png — how WCT and lag_1_oil shape predictions")
print(f"  12_final_dashboard_[well].png — your README hero image")
 
print(f"\n  COMPLETE PROJECT SUMMARY")
print(f"  {'Well':<20} {'Arps RMSE':>10} {'XGB RMSE':>10} {'Winner':>12}")
print(f"  {'-'*55}")
for well_name in TARGET_WELLS:
    arps_r = ARPS_BENCHMARK.get(well_name, {}).get('RMSE', 0)
    xgb_r  = XGB_RESULTS.get(well_name, {}).get('RMSE', 0)
    winner = 'XGBoost' if xgb_r < arps_r else 'Arps DCA'
    diff   = abs(arps_r - xgb_r) / arps_r * 100
    print(f"  {well_name:<20} {arps_r:>10.1f} {xgb_r:>10.1f} {winner:>10} ({diff:.0f}%)")
 
 

Rebuilding data pipeline...
this well has no clean data
this well has no clean data
this well has no clean data
this well has no clean data
Data rebuilt for ['15/9-F-12', '15/9-F-14']

Loading saved XGBoost models...
  Loaded: ../outputs/models/xgb_15-9-F-12.joblib
  Loaded: ../outputs/models/xgb_15-9-F-14.joblib

Computing SHAP values...
  Computing SHAP for 15/9-F-12...
    SHAP matrix shape (train): (73, 19)
    Baseline (expected value): 1705.0 Sm³/day
    This is the average prediction across all training rows.
  Computing SHAP for 15/9-F-14...
    SHAP matrix shape (train): (72, 19)
    Baseline (expected value): 1701.2 Sm³/day
    This is the average prediction across all training rows.

Plot 09: Beeswarm plots...
  Saved → ../outputs/figures/09_shap_beeswarm.png

Plot 10: Waterfall plots (local explanation of best and worst predictions)...
  Saved → ../outputs/figures/10_shap_waterfall_15-9-F-12.png
  Saved → ../outputs/figures/10_shap_waterfall_15-9-F-14.png

Plot 11: SHAP dep

C:\Users\GillA\AppData\Local\Temp\ipykernel_35004\3604465775.py:406: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved → ../outputs/figures/12_final_dashboard_15-9-F-12.png


C:\Users\GillA\AppData\Local\Temp\ipykernel_35004\3604465775.py:406: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved → ../outputs/figures/12_final_dashboard_15-9-F-14.png
